# Activation Capping Qualitative Eval

Given an `activation_capping/` folder on the HF monorepo (e.g. `fine_tuning/llama-3.1-8b-it/ocean/agreeableness/amplifier/vanton4/activation_capping`), this notebook:

1. Downloads `{persona}_axis.pt` + `{persona}_per_layer_range.pt` from that path.
2. Loads the base model recorded in the axis metadata (no LoRA adapter).
3. Picks one prompt from each OCEAN trait file under `data/ocean_open_ended/` (5 total).
4. Generates a response **uncapped** and **capped at fraction=1.0 (floor, toward the LoRA direction)**, using the `recommended_capping_layers` in the axis metadata.
5. Prints the two responses side-by-side per prompt.

In [ ]:
import json
import subprocess
from pathlib import Path

import torch
from dotenv import load_dotenv
from transformers import AutoModelForCausalLM, AutoTokenizer

from src_dev.activation_capping.model import ActivationCappedModel
from src_dev.common.lora_catalogue import HF_REPO
from src_dev.utils.hf_hub import download_from_dataset_repo, login_from_env

load_dotenv()
login_from_env()

REPO_ROOT = Path(subprocess.check_output(["git", "rev-parse", "--show-toplevel"], text=True).strip())

## Config

Point `HF_ACTIVATION_CAPPING_PATH` at the `activation_capping/` folder on the monorepo for the persona you want to inspect. `CAPPING_FRACTION` = 1.0 is floor-at-hi (aggressive push toward the LoRA direction). Flip sign for ceiling (suppress).

In [ ]:
HF_ACTIVATION_CAPPING_PATH = "fine_tuning/llama-3.1-8b-it/ocean/agreeableness/amplifier/vanton4/activation_capping"
CAPPING_FRACTION = 1.0
CEILING_FROM_HI = True  # mirror floor/ceiling semantics (matches eval suite default)

MAX_NEW_TOKENS = 200
TEMPERATURE = 0.0  # deterministic so the comparison is clean
SEED = 42

OCEAN_TRAITS = ["openness", "conscientiousness", "extraversion", "agreeableness", "neuroticism"]
SAMPLE_INDEX = 0  # which row of each open-ended jsonl to use (0..239)

# Derived
_mode = "floor" if CAPPING_FRACTION >= 0 else "ceiling"
persona = HF_ACTIVATION_CAPPING_PATH.rstrip("/").split("/")[-2] + "_" + HF_ACTIVATION_CAPPING_PATH.rstrip("/").split("/")[-3]
# Local cache dir: scratch/activation_capping_qualitative_cache/{slugified path}/
cache_slug = HF_ACTIVATION_CAPPING_PATH.replace("/", "__")
LOCAL_CACHE_DIR = REPO_ROOT / "scratch" / "activation_capping_qualitative_cache" / cache_slug
LOCAL_CACHE_DIR.mkdir(parents=True, exist_ok=True)

print(f"HF path:    {HF_ACTIVATION_CAPPING_PATH}")
print(f"Fraction:   {CAPPING_FRACTION} (mode={_mode})")
print(f"Local cache: {LOCAL_CACHE_DIR}")

In [ ]:
# Download axis + per_layer_range (only the two .pt files we need).
download_from_dataset_repo(
    repo_id=HF_REPO,
    path_in_repo=HF_ACTIVATION_CAPPING_PATH,
    local_dir=LOCAL_CACHE_DIR,
    allow_patterns=["*_axis.pt", "*_per_layer_range.pt"],
)

# HF nests downloads by the repo path. Flatten.
_nested = LOCAL_CACHE_DIR / HF_ACTIVATION_CAPPING_PATH
if _nested.exists():
    for f in _nested.iterdir():
        target = LOCAL_CACHE_DIR / f.name
        if not target.exists():
            f.replace(target)

axis_path = next(LOCAL_CACHE_DIR.glob("*_axis.pt"))
per_layer_range_path = next(LOCAL_CACHE_DIR.glob("*_per_layer_range.pt"))

axis_data = torch.load(axis_path, weights_only=False)
axis_metadata = axis_data["metadata"]
BASE_MODEL = axis_metadata["model"]
CAPPING_LAYERS = list(axis_metadata["recommended_capping_layers"])

print(f"Axis:              {axis_path.name}")
print(f"Per-layer range:   {per_layer_range_path.name}")
print(f"Base model:        {BASE_MODEL}")
print(f"Capping layers:    {CAPPING_LAYERS[0]}-{CAPPING_LAYERS[-1]} (n={len(CAPPING_LAYERS)} of {axis_metadata.get('n_layers', '?')})")

## Load model + tokenizer

In [ ]:
tokenizer = AutoTokenizer.from_pretrained(BASE_MODEL)
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

model = AutoModelForCausalLM.from_pretrained(
    BASE_MODEL,
    torch_dtype=torch.bfloat16,
    device_map="auto",
)
model.eval()
print(f"Loaded {BASE_MODEL} ({len(model.model.layers)} layers)")

In [ ]:
# Pick one prompt from each OCEAN open-ended file.
prompts = []
for trait in OCEAN_TRAITS:
    path = REPO_ROOT / "data" / "ocean_open_ended" / f"{trait}.jsonl"
    with open(path) as f:
        rows = [json.loads(line) for line in f]
    row = rows[SAMPLE_INDEX]
    prompts.append({"trait": trait, "facet": row.get("facet"), "question": row["question"]})

for p in prompts:
    print(f"[{p['trait']}/{p['facet']}]  {p['question']}")

## Generate

In [ ]:
@torch.inference_mode()
def generate(hf_model, question: str) -> str:
    torch.manual_seed(SEED)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(SEED)
    messages = [{"role": "user", "content": question}]
    text = tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
    enc = tokenizer(text, return_tensors="pt", add_special_tokens=False).to(model.device)
    sample_kwargs = {"do_sample": False} if TEMPERATURE == 0.0 else {"do_sample": True, "temperature": TEMPERATURE}
    out = hf_model.generate(
        **enc,
        max_new_tokens=MAX_NEW_TOKENS,
        pad_token_id=tokenizer.pad_token_id,
        **sample_kwargs,
    )
    resp_ids = out[0, enc["input_ids"].shape[1]:]
    return tokenizer.decode(resp_ids, skip_special_tokens=True).strip()

In [ ]:
# Uncapped baseline
print("Generating uncapped responses...")
uncapped_responses = [generate(model, p["question"]) for p in prompts]

In [ ]:
# Capped at fraction=CAPPING_FRACTION
print(f"Generating capped responses (fraction={CAPPING_FRACTION}, mode={_mode})...")
capped_model = ActivationCappedModel.from_pretrained(
    model=model,
    axis_path=str(axis_path),
    per_layer_range_path=str(per_layer_range_path),
    fraction=CAPPING_FRACTION,
    capping_layers=CAPPING_LAYERS,
    mode=_mode,
    ceiling_from_hi=CEILING_FROM_HI,
)
try:
    capped_responses = [generate(capped_model, p["question"]) for p in prompts]
finally:
    capped_model.remove_hooks()
print("Hooks removed.")

## Side-by-side comparison

In [ ]:
for p, uncapped, capped in zip(prompts, uncapped_responses, capped_responses):
    print("=" * 90)
    print(f"[{p['trait'].upper()} / {p['facet']}]  {p['question']}")
    print("-" * 90)
    print("UNCAPPED:")
    print(uncapped)
    print("-" * 90)
    print(f"CAPPED (fraction={CAPPING_FRACTION}, mode={_mode}, layers {CAPPING_LAYERS[0]}-{CAPPING_LAYERS[-1]}):")
    print(capped)
    print()